# Apex LLM - Colab Smoke Test
Run cells from top to bottom. Make sure Runtime is set to T4 GPU first.

In [ ]:
# Cell 1 - GPU check + clone
!nvidia-smi
import torch
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU non detecte. Dans Colab: Execution > Modifier le type d execution > T4 GPU, puis redemarrer la session.')
%cd /content
!rm -rf Apex_LLM
!git clone https://github.com/tovrr/Apex_LLM.git
%cd Apex_LLM

In [ ]:
# Cell 2 - Dependencies
!python -m pip install --upgrade pip
!grep -Ev '^(torch|torchvision|torchaudio)==|^sentence-transformers' requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt
!pip install transformers==5.4.0 peft==0.18.1 trl==1.0.0 accelerate==1.13.0 bitsandbytes==0.49.2
import torch
print('torch =', torch.__version__, 'cuda =', torch.version.cuda)
print('cuda_available =', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('CUDA indisponible apres install. Redemarrer la session Colab puis relancer Cell 1 et Cell 2.')

In [ ]:
# Cell 3 - Smoke train
!python apex_lora.py 2>&1 | tee smoke_train.log

In [ ]:
# Cell 4 - Download LoRA artifact
import os, shutil
# pyright: reportMissingImports=false
from google.colab import files

!find . -name 'adapter_model.safetensors' -type f
!ls -lah apex_lora_final || true

if os.path.isdir('apex_lora_final'):
    shutil.make_archive('apex_lora_final', 'zip', 'apex_lora_final')
    files.download('apex_lora_final.zip')
else:
    print('Dossier apex_lora_final introuvable. Verifie smoke_train.log.')